TASK 03_ GroupBy & Tổng hợp


3. - Đọc dữ liệu sạch 


In [1]:
import pandas as pd

facility = pd.read_csv("../data_clean/facility_info_clean.csv")
incident = pd.read_csv("../data_clean/incident_report_clean.csv")
maintenance = pd.read_csv("../data_clean/maintenance_log_clean.csv")


3.1 - Tính số lượng sự cố theo facility_type

In [2]:
incident_facility_type = (
    incident.merge(
        facility[["facility_id", "facility_type"]],
        on="facility_id",
        how="left"
    )
    .groupby("facility_type")
    .size()
    .reset_index(name="incident_count")
)

incident_facility_type


,facility_type,incident_count
0,Hoi Truong,28
1,Lab,31
2,Phong Hoc,19
3,Phong Hoc,18
4,Studio,24


3.2 - Tính thời gian sửa chữa trung bình theo facility_id

In [5]:
maint_incident = maintenance.merge(
    incident[["report_id", "facility_id"]],
    on="report_id",
    how="left"
)
avg_repair_facility = (
    maint_incident.groupby("facility_id")["duration_hours"]
    .mean()
    .reset_index(name="avg_duration_hours")
)

avg_repair_facility



,facility_id,avg_duration_hours
0,A101,5.000000
1,A102,4.250000
2,A103,3.687500
3,A104,1.500000
4,A105,5.000000
5,A108,2.000000
6,A109,1.500000
7,A114,1.500000
8,A116,NaN
9,A117,3.250000


3.3- Tính thời gian sửa chữa trung bình theo facility_type.

In [8]:
avg_repair_type = (
    maint_incident.merge(
        facility[["facility_id", "facility_type"]],
        on="facility_id",
        how="left"
    )
    .groupby("facility_type")["duration_hours"]
    .mean()
    .reset_index(name="avg_duration_hours")
)


3.4 - Sự cố có thời gian sửa chữa dài nhất theo từng loại phòng

In [7]:
# Nối maintenance → incident → facility
maint_full = (
    maintenance.merge(
        incident[["report_id", "facility_id"]],
        on="report_id",
        how="left"
    )
    .merge(
        facility[["facility_id", "facility_type"]],
        on="facility_id",
        how="left"
    )
)

# Lấy sự cố có thời gian sửa chữa dài nhất theo từng facility_type
longest_repair_by_type = (
    maint_full.loc[
        maint_full.groupby("facility_type")["duration_hours"].idxmax()
    ]
)

longest_repair_by_type



,log_id,report_id,maintenance_date,duration_hours,technician,facility_id,facility_type
17,M7017,R3030,2024-01-10,5.00,Nguyen Van A,A145,Hoi Truong
0,M7000,R3051,2024-03-30,5.00,Nguyen Van A,A124,Lab
47,M7047,R3108,2024-05-09,4.25,Nguyen Van A,A134,Phong Hoc
50,M7050,R3075,NaN,5.00,Le Thi B,A157,Phong Hoc
64,M7064,R3069,NaN,5.00,Nguyen Van A,A136,Studio


3.5 - Bảng tổng hợp đánh giá bảo trì theo loại CSVC

In [10]:
# Nối maintenance → incident → facility
maint_full = (
    maintenance.merge(
        incident[["report_id", "facility_id"]],
        on="report_id",
        how="left"
    )
    .merge(
        facility[["facility_id", "facility_type"]],
        on="facility_id",
        how="left"
    )
)

# Bảng tổng hợp đánh giá đặc trưng bảo trì theo loại CSVC
summary_by_type = (
    maint_full.groupby("facility_type")
    .agg(
        total_incidents=("report_id", "count"),
        avg_repair_time=("duration_hours", "mean"),
        max_repair_time=("duration_hours", "max"),
        min_repair_time=("duration_hours", "min")
    )
    .reset_index()
)

summary_by_type



,facility_type,total_incidents,avg_repair_time,max_repair_time,min_repair_time
0,Hoi Truong,24,3.426471,5.00,1.5
1,Lab,29,3.112500,5.00,1.5
2,Phong Hoc,14,2.093750,4.25,1.5
3,Phong Hoc,18,3.153846,5.00,1.5
4,Studio,15,3.075000,5.00,1.5
